## Imports

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Preprocessing
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import OrdinalEncoder
from sklearn.preprocessing import MinMaxScaler

# Classification
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.naive_bayes import CategoricalNB
from sklearn.linear_model import LogisticRegression

# Verification and Summary
from sklearn.metrics import f1_score
from sklearn.metrics import accuracy_score
from sklearn.metrics import precision_score
from sklearn.metrics import recall_score
from sklearn.metrics import confusion_matrix
from sklearn.metrics import classification_report

# Pickling
import pickle

# LCN Python package
from hiclass import LocalClassifierPerNode
from hiclass import metrics

# JSON
import json

# Progress bar
from tqdm import tqdm
import time

# Ignore warnings
# warnings.simplefilter(action='ignore', category=FutureWarning)
import warnings
warnings.filterwarnings("ignore")

## Validation

In [4]:
sig_to_evttagid = pd.read_csv('Inputs/sig-evttags-map-02.csv')
display(sig_to_evttagid.head(5))

df = pd.read_csv('Inputs/pmu_transmission_features.csv')
display(df.head(5))

sec_to_tert_dict = json.load(open('Inputs/sec_to_tert_dict_transmission.json'))

df_dist = pd.read_csv('Inputs/pmu_distribution_features.csv')
df_dist_2_3 = df_dist.loc[df_dist['sigid'] < 655].copy()
df_dist_4 = df_dist.loc[df_dist['sigid'] >= 655].copy().loc[:, ~df_dist.columns.isin(['mean_v', 'variance_v', 'skewness_v', 'kurtosis_v'])]
display(df_dist.head(5))

,sigid,event_tag_id,evttag,primary,secondary,tertiary
0,967,279,Equipment::Interrupting Device::Circuit Breaker,Equipment,Interrupting Device,Circuit Breaker
1,967,137,Events::Other::Unknown,Events,Other,Unknown
2,967,185,Phase::Three-phase::Phases ABC,Phase,Three-phase,Phases ABC
3,970,21,Equipment::Interrupting Device,Equipment,Interrupting Device,NaN
4,970,264,Equipment::Overhead Equipment::Arrester,Equipment,Overhead Equipment,Arrester


,sigid,primary,secondary,tertiary,P001.df_mean,P001.df_std,P001.df_skew,P001.df_kurtosis,P001.f_mean,P001.f_std,...,P001.ip_m_skew,P001.ip_m_kurtosis,P001.vp_a_mean,P001.vp_a_std,P001.vp_a_skew,P001.vp_a_kurtosis,P001.vp_m_mean,P001.vp_m_std,P001.vp_m_skew,P001.vp_m_kurtosis
0,974,Equipment,Substation,Transmission,-9.780000e-17,1.000069,1.636972,815.653017,5.040000e-13,1.000069,...,0.211944,1.337818,1.210000e-15,1.000069,0.025759,-1.186279,1.410000e-14,1.000069,-24.240373,1122.540917
1,974,State,Change-State,Trip,-9.780000e-17,1.000069,1.636972,815.653017,5.040000e-13,1.000069,...,0.211944,1.337818,1.210000e-15,1.000069,0.025759,-1.186279,1.410000e-14,1.000069,-24.240373,1122.540917
2,975,Equipment,Overhead-Equipment,Conductor,4.430000e-17,1.000069,2.290180,206.810139,-2.060000e-13,1.000069,...,19.315180,605.768682,1.440000e-15,1.000069,0.194046,-1.328212,2.610000e-14,1.000069,-28.084094,1014.696433
3,975,State,Change-State,Trip,4.430000e-17,1.000069,2.290180,206.810139,-2.060000e-13,1.000069,...,19.315180,605.768682,1.440000e-15,1.000069,0.194046,-1.328212,2.610000e-14,1.000069,-28.084094,1014.696433
4,976,Equipment,Overhead-Equipment,Conductor,3.730000e-18,1.000069,0.821341,48.849595,-3.640000e-14,1.000069,...,0.771972,8.043328,1.220000e-15,1.000069,0.341874,-0.788582,1.740000e-13,1.000069,-18.540243,729.983718


,sigid,primary,secondary,tertiary,mean_v,variance_v,skewness_v,kurtosis_v,mean_f,variance_f,skewness_f,kurtosis_f,mean_pa,variance_pa,skewness_pa,kurtosis_pa,mean_rocof,variance_rocof,skewness_rocof,kurtosis_rocof
0,317,Events,Fault,Unknown Fault,-0.000246,1.000502,-12.638799,206.466832,0.000053,1.000554,0.549123,1.194993,0.000685,1.000134,0.041727,-1.723707,7.594963e-18,1.000278,-6.288910,135.200714
1,318,Events,Fault,Unknown Fault,-0.000038,1.000555,-3.140139,28.746609,0.000704,1.000110,0.020678,-0.977754,-0.001382,0.998838,0.777411,-0.078019,-1.907955e-17,1.000278,-4.199705,177.514052
2,319,Events,Fault,Unknown Fault,0.000215,1.000515,-12.775239,342.246209,0.000650,1.000177,-0.459170,-1.495513,0.000881,0.999858,0.147293,-1.257150,3.547403e-17,1.000278,-1.632768,57.442791
3,320,Events,Fault,Unknown Fault,-0.000047,1.000554,-5.580466,53.925557,0.000160,1.000533,-0.048394,0.182672,0.000785,1.000002,0.170858,-1.375886,-7.687584e-18,1.000278,15.350096,756.468125
4,321,Events,Fault,Unknown Fault,0.000169,1.000531,-3.783521,46.652683,-0.000294,1.000478,-3.292097,21.642079,-0.000941,0.999760,-0.027294,-1.094721,-1.168575e-17,1.000278,0.026002,204.894604


### Validation using hiClass package

In [33]:
primary_arr = ['Events', 'State', 'Equipment', 'Conditions', 'Phase']
precision = []
recall = []
f1 = []

# RF model only
# test_results_all = pd.read_csv("Outputs/pred_to_true_labels_results.csv")

# NB model only
# test_results_all = pd.read_csv("Outputs/pred_to_true_labels_results_NB.csv")

# SVC (RBF) only
test_results_all = pd.read_csv("Outputs/pred_to_true_labels_results_SVC.csv")

for primary_event in primary_arr:
    test_results_subset = test_results_all.loc[test_results_all.Primary == primary_event]
    print(test_results_subset.shape)
    y_pred = test_results_subset[['Predicted_Secondary', 'Predicted_Tertiary']].to_numpy()
    y_true = test_results_subset[['Actual_Secondary', 'Actual_Tertiary']].to_numpy()
        
    precision.append(metrics.precision(y_true, y_pred))
    recall.append(metrics.recall(y_true, y_pred))
    f1.append(metrics.f1(y_true, y_pred))

results_manual = pd.DataFrame(columns=['Primary', 'hP', 'hR', 'hF'])
results_manual['Primary'] = primary_arr
results_manual['hP'] = precision
results_manual['hR'] = recall
results_manual['hF'] = f1
display(results_manual)

(370, 5)
(596, 5)
(304, 5)
(79, 5)
(50, 5)


,Primary,hP,hR,hF
0,Events,0.610811,0.610811,0.610811
1,State,0.969799,0.969799,0.969799
2,Equipment,0.815789,0.815789,0.815789
3,Conditions,0.740506,0.740506,0.740506
4,Phase,0.600000,0.600000,0.600000


### Manual validation (still needs a recheck)

In [20]:
# Attempt in calculating hierarchical precision (hP) and recall (hR), aka hierarchy-based measures, manually
# Results are a bit off from what the HiClass Python package but might be because of how the result format
# Source: Kiritchenko et al., 2005

# Getting key from values: 
# list(mydict.keys())[list(mydict.values()).index(16)] 
# where n in index(n) is the value that you want to get the key of

# Getting intersection of two arrays:
# Source: https://stackoverflow.com/questions/47276494/python-return-intersection-of-two-arrays
def intersection(list_a, list_b):
   return [ e for e in list_a if e in list_b ]

# hP = 0
# hR = 0
# hF = 0
beta = 1

for primary_event in primary_arr:
    test_results_all_subset = test_results_all[test_results_all['Primary'] == primary_event].reset_index()
    hP_num = 0
    hP_denom = 0
    hR_num = 0
    hR_denom = 0
    
    for i in range(0, test_results_all_subset.shape[0]):
        C_i = [] # actual tertiary class
        C_i_hat = [] # ancestors of C_i 
        C_i_prime = [] # predicted tertiary class
        C_i_prime_hat = [] # ancestors of C_i_prime

        C_i.append(test_results_all['Actual_Tertiary'][i])

        C_i_hat.append(test_results_all['Actual_Secondary'][i])
        C_i_hat.append(test_results_all['Actual_Tertiary'][i])

        C_i_prime.append(test_results_all['Predicted_Tertiary'][i])

        C_i_prime_hat.append(test_results_all['Predicted_Secondary'][i])
        C_i_prime_hat.append(test_results_all['Predicted_Tertiary'][i])
        
#         print(C_i_prime_hat)
#         print(f"{C_i_hat}\n")

        hP_num += len(intersection(C_i_hat, C_i_prime_hat))
        hP_denom += len(C_i_prime_hat)
        hR_num += len(intersection(C_i_hat, C_i_prime_hat))
        hR_denom += len(C_i_hat)
        
    hP = hP_num / hP_denom * 100
    hR = hR_num / hR_denom * 100
    hF = ((beta**2 + 1) * hP * hR) / (beta**2 * hP + hR) 

    print(f"The hierarchical precision for {primary_event} is {hP}%.\n")
    print(f"The hierarchical recall for {primary_event} is {hR}%.\n")
    print(f"The hierarchical F-1 score for {primary_event} is {hF}%.\n\n")

The hierarchical precision for Events is 80.67567567567568%.

The hierarchical recall for Events is 80.67567567567568%.

The hierarchical F-1 score for Events is 80.67567567567568%.


The hierarchical precision for State is 88.00335570469798%.

The hierarchical recall for State is 88.00335570469798%.

The hierarchical F-1 score for State is 88.00335570469798%.


The hierarchical precision for Equipment is 79.76973684210526%.

The hierarchical recall for Equipment is 79.76973684210526%.

The hierarchical F-1 score for Equipment is 79.76973684210526%.


The hierarchical precision for Conditions is 79.74683544303798%.

The hierarchical recall for Conditions is 79.74683544303798%.

The hierarchical F-1 score for Conditions is 79.74683544303798%.


The hierarchical precision for Phase is 83.0%.

The hierarchical recall for Phase is 83.0%.

The hierarchical F-1 score for Phase is 83.0%.




## Classification (No need to rerun this section)

In [3]:
sec_to_tert_dict

{'System-Event-Condition': ['Frequency-Deviation',
  'Oscillation',
  'Nonspecific-System-Event-Condition'],
 'External': ['Lightning-Strike', 'Nonspecific-External'],
 'Equipment-Failure': ['Nonspecific-Equipment-Failure', 'Transformer-Failure'],
 'Fault': ['Nonspecific-Fault'],
 'Construction-&-Maintenance': ['Maintenance',
  'Accident',
  'Nonspecific-Construction-&-Maintenance'],
 'Switching-Event': ['Nonspecific-Switching-Event'],
 'Vegetation': ['Tree'],
 'Other': ['Fire'],
 'Defect-or-Deterioration': ['Contamination'],
 'Wildlife': ['Nonspecific-Wildlife'],
 'Protection-Event': ['Nonspecific-Protection-Event'],
 'Steady-State': ['Nonspecific-Steady-State'],
 'Change-State': ['Trip'],
 'Overhead-Equipment': ['Conductor',
  'Nonspecific-Overhead-Equipment',
  'Arrester'],
 'Sources': ['Generator', 'Wind'],
 'Interrupting-Device': ['Nonspecific-Interrupting-Device'],
 'Transformer': ['Nonspecific-Transformer'],
 'Substation': ['Transmission'],
 'Devices': ['Capacitor'],
 'Weather':

### This function is essentially the manual hierarchical classifier
Assumes that training is already completed and that trained classifiers are pickled. Note the specific naming format for the pickled classifiers.

In [21]:
# data_source = ['Transmission_PMU', 'Distribution_PMU_2_3', 'Distribution_PMU_4', 'POW', 'Lab_Testing']
# classifier = ['NB', 'RF', 'SVC' (which is RBF, i.e. default for sklearn), SVC_Linear', 'SVC_Polynomial', 'Logit']
def successiveBinaryClassification(test_features, label_list, data_source, classifier):
    proba_temp = 0
    pred_class = ''
    
    for evttag in label_list:
        fname = classifier + '_' + evttag
        # Change file
        with open(f'Classifiers/{data_source}/{classifier}_{evttag}.pickle', 'rb') as handle:
            globals()[fname] = pickle.load(handle)
            
        if classifier == 'NB':
            scaler = MinMaxScaler()
            test_features = scaler.fit_transform(test_features)

        proba = globals()[fname].predict_proba(test_features)
        if proba[0][1] > proba_temp:
            proba_temp = proba[0][1]
            pred_class = evttag
    
    return pred_class

In [27]:
primary_arr = ['Events', 'State', 'Equipment', 'Conditions', 'Phase']
acc = []
primary_all = []
actual_class_sec_all = []
pred_class_sec_all = []
actual_class_tert_all = []
pred_class_tert_all = []

for primary_event in primary_arr:
    print(f'Classifying for testing dataset for {primary_event}...\n')
    df1 = df.copy().loc[df['primary'] == primary_event].dropna().reset_index()

    # Unique tertiary labels ordered from highest count number to lowest count number
    arr = np.asanyarray(df1.groupby(['secondary']).nunique().sort_values(by=['index'], ascending=False).index)
    arr_tertiary = np.asanyarray(df1.groupby(['tertiary']).nunique().sort_values(by=['index'], ascending=False).index)
    features = df1.iloc[:, -24:-1]
    label = df1['secondary']

    x_train, x_test, y_train, y_test = train_test_split(features, label, test_size=0.05, random_state=42)
    # Classify each entry in the testing dataset individually and print out the RF classifier
    # 'True' label that outputs the highest probability.
    actual_class_sec = []
    pred_class_sec = []
    actual_class_tert = []
    pred_class_tert = []

    for idx in x_test.index:
        primary_all.append(primary_event)
#         print(f'Classifying secondary label for row {idx}')
        test_features = x_test[x_test.index == idx]
        pred_class = successiveBinaryClassification(test_features, arr, 'Transmission_PMU', 'SVC')
        actual_class_sec.append(df1.iloc[idx]['secondary'])
        pred_class_sec.append(pred_class)
        actual_class_sec_all.append(df1.iloc[idx]['secondary'])
        pred_class_sec_all.append(pred_class)

#         print(f'Classifying tertiary label for row {idx}\n')
        pred_class = successiveBinaryClassification(test_features, sec_to_tert_dict[pred_class], 'Transmission_PMU', 'SVC')
        actual_class_tert.append(df1.iloc[idx]['tertiary'])
        pred_class_tert.append(pred_class)
        actual_class_tert_all.append(df1.iloc[idx]['tertiary'])
        pred_class_tert_all.append(pred_class)

    test_results = pd.DataFrame(columns=['Actual_Secondary', 'Predicted_Secondary', 
                                         'Actual_Tertiary', 'Predicted_Tertiary'])
    test_results['Actual_Secondary'] = actual_class_sec
    test_results['Predicted_Secondary'] = pred_class_sec
    test_results['Actual_Tertiary'] = actual_class_tert
    test_results['Predicted_Tertiary'] = pred_class_tert
    
    correct_classification_sec = test_results.loc[test_results['Actual_Secondary'] == test_results['Predicted_Secondary']].shape[0]
    correct_classification_tert = test_results.loc[test_results['Actual_Tertiary'] == test_results['Predicted_Tertiary']].shape[0]
    acc.append(correct_classification_tert / test_results.shape[0] * 100)

test_results_all = pd.DataFrame(columns=['Primary', 'Actual_Secondary', 'Predicted_Secondary', 
                                         'Actual_Tertiary', 'Predicted_Tertiary'])
test_results_all['Primary'] = primary_all
test_results_all['Actual_Secondary'] = actual_class_sec_all
test_results_all['Predicted_Secondary'] = pred_class_sec_all
test_results_all['Actual_Tertiary'] = actual_class_tert_all
test_results_all['Predicted_Tertiary'] = pred_class_tert_all


results = pd.DataFrame(columns=['primary', 'acc'])
results['primary'] = primary_arr
results['acc'] = acc
results

Classifying for testing dataset for Events...

Classifying for testing dataset for State...

Classifying for testing dataset for Equipment...

Classifying for testing dataset for Conditions...

Classifying for testing dataset for Phase...



,primary,acc
0,Events,59.459459
1,State,96.979866
2,Equipment,78.618421
3,Conditions,48.101266
4,Phase,40.000000


In [6]:
test_results_all[test_results_all.Primary == 'Equipment']

,Primary,Actual_Secondary,Predicted_Secondary,Actual_Tertiary,Predicted_Tertiary
966,Equipment,Overhead-Equipment,Overhead-Equipment,Conductor,Conductor
967,Equipment,Overhead-Equipment,Overhead-Equipment,Conductor,Conductor
968,Equipment,Overhead-Equipment,Overhead-Equipment,Conductor,Conductor
969,Equipment,Overhead-Equipment,Overhead-Equipment,Conductor,Nonspecific-Overhead-Equipment
970,Equipment,Overhead-Equipment,Overhead-Equipment,Conductor,Conductor
...,...,...,...,...,...
1265,Equipment,Overhead-Equipment,Overhead-Equipment,Conductor,Conductor
1266,Equipment,Overhead-Equipment,Overhead-Equipment,Conductor,Conductor
1267,Equipment,Sources,Sources,Generator,Generator
1268,Equipment,Overhead-Equipment,Overhead-Equipment,Conductor,Conductor


In [28]:
# Saving classification results to two csv files so that there is no need to 
# rerun individual tests when calculating validation metrics
# results.to_csv("Outputs/acc_results.csv", index=False)
test_results_all.to_csv("Outputs/pred_to_true_labels_results_SVC.csv", index=False)

## No need to re-run below unless new classifiers must be trained. 

In [5]:
# Insert which df has features and statistical measures, then run all the code blocks below
df_master = df

In [6]:
# Make a dictionary that stores an array of tertiary labels for each secondary label.
# This is for transmission PMU.
sec_to_tert_dict = {}

primary_arr = ['Events', 'State', 'Equipment', 'Conditions', 'Phase']

for primary_event in primary_arr:
    df1 = df_master.copy().loc[df_master['primary'] == primary_event].dropna().reset_index()

    # Unique tertiary labels ordered from highest count number to lowest count number
    arr = np.asanyarray(df1.groupby(['secondary']).nunique().sort_values(by=['index'], ascending=False).index)
    arr_tertiary = np.asanyarray(df1.groupby(['tertiary']).nunique().sort_values(by=['index'], ascending=False).index)
    for evttag in arr:
        temp = df_master.copy().loc[df_master['secondary'] == evttag].dropna().reset_index()
        temp_arr = (np.asanyarray(temp.groupby(['tertiary']).nunique().sort_values(by=['index'], ascending=False).index)).tolist()
        sec_to_tert_dict[evttag] = temp_arr
        
print(sec_to_tert_dict)
    
# # ============= CHANGE SAVE FILE NAME ============= #
# with open('sec_to_tert_dict_provider_4.json', 'w') as outfile:
#     json.dump(sec_to_tert_dict, outfile)

{'System-Event-Condition': ['Frequency-Deviation', 'Oscillation', 'Nonspecific-System-Event-Condition'], 'External': ['Lightning-Strike', 'Nonspecific-External'], 'Equipment-Failure': ['Nonspecific-Equipment-Failure', 'Transformer-Failure'], 'Fault': ['Nonspecific-Fault'], 'Construction-&-Maintenance': ['Maintenance', 'Accident', 'Nonspecific-Construction-&-Maintenance'], 'Switching-Event': ['Nonspecific-Switching-Event'], 'Vegetation': ['Tree'], 'Other': ['Fire'], 'Defect-or-Deterioration': ['Contamination'], 'Wildlife': ['Nonspecific-Wildlife'], 'Protection-Event': ['Nonspecific-Protection-Event'], 'Steady-State': ['Nonspecific-Steady-State'], 'Change-State': ['Trip'], 'Overhead-Equipment': ['Conductor', 'Nonspecific-Overhead-Equipment', 'Arrester'], 'Sources': ['Generator', 'Wind'], 'Interrupting-Device': ['Nonspecific-Interrupting-Device'], 'Transformer': ['Nonspecific-Transformer'], 'Substation': ['Transmission'], 'Devices': ['Capacitor'], 'Weather': ['Lightning-Storm', 'Nonspecif

### Naïve (Categorical) Bayesian
Important to note that MinMaxScaler is used because the NB classifier does not take negative values in feature set.

In [ ]:
for primary_events in primary_arr:
    df1 = df_master.copy().loc[df_master['primary'] == primary_events].dropna().reset_index()

    arr = np.asanyarray(df1.groupby(['secondary']).nunique().sort_values(by=['index'], ascending=False).index)
    arr_tertiary = np.asanyarray(df1.groupby(['tertiary']).nunique().sort_values(by=['index'], ascending=False).index)
    
    if df1.shape[0] != 0:
        # ============= SECONDARY LABEL TRAINING ============= #
        features = df1.iloc[:, -(df1.shape[1] - 5):-1]
        label = df1['secondary']

        x_train, x_test, y_train, y_test = train_test_split(features, label, test_size=0.05, random_state=42)

        for evttag in arr:
            print(f'Training (Secondary) {evttag} binary classifier...')
            y_train_copy = y_train == evttag
            globals()['NB_%s' % evttag] = CategoricalNB()
            scaler = MinMaxScaler()
            x_train = scaler.fit_transform(x_train)
            globals()['NB_%s' % evttag].fit(x_train, y_train_copy.values)
            
            print(f'Pickling NB classifier for {evttag}...\n')
            # ============= CHANGE SAVE FILE PATH ============= #
            with open(f'Classifiers/Transmission_PMU/NB_{evttag}.pickle', 'wb') as handle:
                pickle.dump(globals()['NB_%s' % evttag], handle, protocol=pickle.HIGHEST_PROTOCOL)

        # ============= TERTIARY LABEL TRAINING ============= #
        features = df1.iloc[:, -(df1.shape[1] - 5):-1]
        label = df1['tertiary']

        x_train, x_test, y_train, y_test = train_test_split(features, label, test_size=0.05, random_state=42)

        for evttag in arr_tertiary:
            print(f'Training (Tertiary) {evttag} binary classifier...')
            y_train_copy = y_train == evttag
            globals()['NB_%s' % evttag] = CategoricalNB()
            scaler = MinMaxScaler()
            x_train = scaler.fit_transform(x_train)
            globals()['NB_%s' % evttag].fit(x_train, y_train_copy.values)

            print(f'Pickling NB classifier for {evttag}...\n')
            # ============= CHANGE SAVE FILE PATH ============= #
            with open(f'Classifiers/Transmission_PMU/NB_{evttag}.pickle', 'wb') as handle:
                pickle.dump(globals()['NB_%s' % evttag], handle, protocol=pickle.HIGHEST_PROTOCOL)

### Support Vector Machine
Need to set probability=True so that probabilities will be outputted by classifier. 

In [ ]:
kernel = 'linear'
for primary_events in primary_arr:
    df1 = df_master.copy().loc[df_master['primary'] == primary_events].dropna().reset_index()

    arr = np.asanyarray(df1.groupby(['secondary']).nunique().sort_values(by=['index'], ascending=False).index)
    arr_tertiary = np.asanyarray(df1.groupby(['tertiary']).nunique().sort_values(by=['index'], ascending=False).index)
    
    if df1.shape[0] != 0:
        # ============= SECONDARY LABEL TRAINING ============= #
        features = df1.iloc[:, -(df1.shape[1] - 5):-1]
        label = df1['secondary']

        x_train, x_test, y_train, y_test = train_test_split(features, label, test_size=0.05, random_state=42)

        for evttag in arr:
            print(f'Training (Secondary) {evttag} binary classifier...')
            y_train_copy = y_train == evttag
            globals()['SVC_Linear_%s' % evttag] = SVC(kernel=kernel, probability=True)
            globals()['SVC_Linear_%s' % evttag].fit(x_train, y_train_copy.values)
            
            print(f'Pickling SVC classifier for {evttag}...\n')
            # ============= CHANGE SAVE FILE PATH ============= #
            with open(f'Classifiers/Transmission_PMU/SVC_Linear_{evttag}.pickle', 'wb') as handle:
                pickle.dump(globals()['SVC_Linear_%s' % evttag], handle, protocol=pickle.HIGHEST_PROTOCOL)

        # ============= TERTIARY LABEL TRAINING ============= #
        features = df1.iloc[:, -(df1.shape[1] - 5):-1]
        label = df1['tertiary']

        x_train, x_test, y_train, y_test = train_test_split(features, label, test_size=0.05, random_state=42)

        for evttag in arr_tertiary:
            print(f'Training (Tertiary) {evttag} binary classifier...')
            y_train_copy = y_train == evttag
            globals()['SVC_Linear_%s' % evttag] = SVC(kernel=kernel, probability=True)
            globals()['SVC_Linear_%s' % evttag].fit(x_train, y_train_copy.values)

            print(f'Pickling SVC classifier for {evttag}...\n')
            # ============= CHANGE SAVE FILE PATH ============= #
            with open(f'Classifiers/Transmission_PMU/SVC_Linear_{evttag}.pickle', 'wb') as handle:
                pickle.dump(globals()['SVC_Linear_%s' % evttag], handle, protocol=pickle.HIGHEST_PROTOCOL)

Training (Secondary) System-Event-Condition binary classifier...
Pickling SVC classifier for System-Event-Condition...

Training (Secondary) External binary classifier...
Pickling SVC classifier for External...

Training (Secondary) Equipment-Failure binary classifier...


### Random Forest

In [ ]:
n_estimators = 1000
max_depth = 8

for primary_event in primary_arr:
    df1 = df_master.copy().loc[df_master['primary'] == primary_event].dropna().reset_index()

    arr = np.asanyarray(df1.groupby(['secondary']).nunique().sort_values(by=['index'], ascending=False).index)
    arr_tertiary = np.asanyarray(df1.groupby(['tertiary']).nunique().sort_values(by=['index'], ascending=False).index)
    
    if df1.shape[0] != 0:
        # ============= SECONDARY LABEL TRAINING ============= #
        features = df1.iloc[:, -(df1.shape[1] - 5):-1]
        label = df1['secondary']

        x_train, x_test, y_train, y_test = train_test_split(features, label, test_size=0.05, random_state=42)

        for evttag in arr:
            print(f'Training (Secondary) {evttag} binary classifier...')
            y_train_copy = y_train == evttag
            globals()['RF_%s' % evttag] = RandomForestClassifier(n_estimators=n_estimators, 
                                                                 max_depth=max_depth, random_state=42)
            globals()['RF_%s' % evttag].fit(x_train, y_train_copy.values)

            print(f'Pickling RF classifier for {evttag}...\n')
            # ============= CHANGE SAVE FILE PATH ============= #
            with open(f'Classifiers/Transmission_PMU/RF_{evttag}.pickle', 'wb') as handle:
                pickle.dump(globals()['RF_%s' % evttag], handle, protocol=pickle.HIGHEST_PROTOCOL)

        # ============= TERTIARY LABEL TRAINING ============= #
        features = df1.iloc[:, -(df1.shape[1] - 5):-1]
        label = df1['tertiary']

        x_train, x_test, y_train, y_test = train_test_split(features, label, test_size=0.05, random_state=42)

        for evttag in arr_tertiary:
            print(f'Training (Tertiary) {evttag} binary classifier...')
            y_train_copy = y_train == evttag
            globals()['RF_%s' % evttag] = RandomForestClassifier(n_estimators=n_estimators, 
                                                                 max_depth=max_depth, random_state=42)
            globals()['RF_%s' % evttag].fit(x_train, y_train_copy.values)

            print(f'Pickling RF classifier for {evttag}...\n')
            # ============= CHANGE SAVE FILE PATH ============= #
            with open(f'Classifiers/Transmission_PMU/RF_{evttag}.pickle', 'wb') as handle:
                pickle.dump(globals()['RF_%s' % evttag], handle, protocol=pickle.HIGHEST_PROTOCOL)

### Regression

In [11]:
one_tertiary_evttag = []

for primary_event in primary_arr:
    df1 = df_master.copy().loc[df_master['primary'] == primary_event].dropna().reset_index()

    arr = np.asanyarray(df1.groupby(['secondary']).nunique().sort_values(by=['index'], ascending=False).index)
    arr_tertiary = np.asanyarray(df1.groupby(['tertiary']).nunique().sort_values(by=['index'], ascending=False).index)
    
    if df1.shape[0] != 0:
        # ============= SECONDARY LABEL TRAINING ============= #
        features = df1.iloc[:, -(df1.shape[1] - 5):-1]
        label = df1['secondary']

        x_train, x_test, y_train, y_test = train_test_split(features, label, test_size=0.05, random_state=42)
        idx = x_train.index
#         print(idx)
#         display(x_train)

        for i in tqdm (range (100), desc=f'Fitting Secondary logit under {primary_event}...'):
            globals()['Logit_Secondary_Under_%s' % primary_event] = LogisticRegression(multi_class='ovr', max_iter=3000)
            globals()['Logit_Secondary_Under_%s' % primary_event].fit(x_train, y_train.values)

        for i in tqdm (range (100), desc=f'Pickling Secondary logit under {primary_event}...'):
            # ============= CHANGE SAVE FILE PATH ============= #
            with open(f'Classifiers/Transmission_PMU/Logit_Secondary_Under_{primary_event}.pickle', 'wb') as handle:
                pickle.dump(globals()['Logit_Secondary_Under_%s' % primary_event], handle, protocol=pickle.HIGHEST_PROTOCOL)
                
        for evttag in arr:
            df2 = df1.copy().loc[df1['secondary'] == evttag].dropna()
        
            # Make sure to train on only data within the training dataset
            df2 = df2[df2.index.isin(idx)]
            # display(df2)
            
            features_tert = df2.iloc[:, -(df2.shape[1] - 5):-1]
            # display(features_tert)
            label_tert = df2['tertiary']
            
            for i in tqdm (range (100), desc=f'Fitting Tertiary logit under {evttag}...'):
                globals()['Logit_Tertiary_Under_%s' % evttag] = LogisticRegression(multi_class='ovr', max_iter=3000)
                try:
                    globals()['Logit_Tertiary_Under_%s' % evttag].fit(features_tert, label_tert.values)
                    # ============= CHANGE SAVE FILE PATH ============= #
                    with open(f'Classifiers/Transmission_PMU/Logit_Tertiary_Under_{evttag}.pickle', 'wb') as handle:
                        pickle.dump(globals()['Logit_Tertiary_Under_%s' % evttag], handle, protocol=pickle.HIGHEST_PROTOCOL)
                except ValueError:
                    one_tertiary_evttag.append(evttag)
                    print(f'No regression model fitted because {evttag} only has one tertiary label.')
                    continue

Pickling Secondary logit under Events...: 100%|██████████| 100/100 [00:00<00:00, 551.59it/s]
Fitting Tertiary logit under System-Event-Condition...: 100%|██████████| 100/100 [03:22<00:00,  2.03s/it]
Fitting Tertiary logit under External...: 100%|██████████| 100/100 [00:45<00:00,  2.19it/s]
Fitting Tertiary logit under Equipment-Failure...: 100%|██████████| 100/100 [01:09<00:00,  1.44it/s]
Fitting Tertiary logit under Fault...: 100%|██████████| 100/100 [00:00<00:00, 565.91it/s]


No regression model fitted because Fault only has one tertiary label.
No regression model fitted because Fault only has one tertiary label.
No regression model fitted because Fault only has one tertiary label.
No regression model fitted because Fault only has one tertiary label.
No regression model fitted because Fault only has one tertiary label.
No regression model fitted because Fault only has one tertiary label.
No regression model fitted because Fault only has one tertiary label.
No regression model fitted because Fault only has one tertiary label.
No regression model fitted because Fault only has one tertiary label.
No regression model fitted because Fault only has one tertiary label.
No regression model fitted because Fault only has one tertiary label.
No regression model fitted because Fault only has one tertiary label.
No regression model fitted because Fault only has one tertiary label.
No regression model fitted because Fault only has one tertiary label.
No regression model 

Fitting Tertiary logit under Construction-&-Maintenance...: 100%|██████████| 100/100 [00:42<00:00,  2.35it/s]
Fitting Tertiary logit under Switching-Event...: 100%|██████████| 100/100 [00:00<00:00, 614.45it/s]


No regression model fitted because Switching-Event only has one tertiary label.
No regression model fitted because Switching-Event only has one tertiary label.
No regression model fitted because Switching-Event only has one tertiary label.
No regression model fitted because Switching-Event only has one tertiary label.
No regression model fitted because Switching-Event only has one tertiary label.
No regression model fitted because Switching-Event only has one tertiary label.
No regression model fitted because Switching-Event only has one tertiary label.
No regression model fitted because Switching-Event only has one tertiary label.
No regression model fitted because Switching-Event only has one tertiary label.
No regression model fitted because Switching-Event only has one tertiary label.
No regression model fitted because Switching-Event only has one tertiary label.
No regression model fitted because Switching-Event only has one tertiary label.
No regression model fitted because Switc

Fitting Tertiary logit under Vegetation...:   0%|          | 0/100 [00:00<?, ?it/s]

...

Fitting Tertiary logit under Single-phase...:   0%|          | 0/100 [00:00<?, ?it/s]

No regression model fitted because Single-phase only has one tertiary label.
No regression model fitted because Single-phase only has one tertiary label.
No regression model fitted because Single-phase only has one tertiary label.
No regression model fitted because Single-phase only has one tertiary label.
No regression model fitted because Single-phase only has one tertiary label.
No regression model fitted because Single-phase only has one tertiary label.
No regression model fitted because Single-phase only has one tertiary label.
No regression model fitted because Single-phase only has one tertiary label.
No regression model fitted because Single-phase only has one tertiary label.
No regression model fitted because Single-phase only has one tertiary label.
No regression model fitted because Single-phase only has one tertiary label.
No regression model fitted because Single-phase only has one tertiary label.
No regression model fitted because Single-phase only has one tertiary label.

Fitting Tertiary logit under Single-phase...: 100%|██████████| 100/100 [00:00<00:00, 611.20it/s]



No regression model fitted because Single-phase only has one tertiary label.
No regression model fitted because Single-phase only has one tertiary label.
No regression model fitted because Single-phase only has one tertiary label.
No regression model fitted because Single-phase only has one tertiary label.
No regression model fitted because Single-phase only has one tertiary label.
No regression model fitted because Single-phase only has one tertiary label.
No regression model fitted because Single-phase only has one tertiary label.
No regression model fitted because Single-phase only has one tertiary label.
No regression model fitted because Single-phase only has one tertiary label.
No regression model fitted because Single-phase only has one tertiary label.
No regression model fitted because Single-phase only has one tertiary label.
No regression model fitted because Single-phase only has one tertiary label.
No regression model fitted because Single-phase only has one tertiary label

Fitting Tertiary logit under Two-phase-ground...: 100%|██████████| 100/100 [00:52<00:00,  1.89it/s]


In [11]:
# Regression validation
primary_arr = ['Events', 'State', 'Equipment', 'Conditions', 'Phase']
acc = []

for primary_event in primary_arr:
    df1 = df_master.copy().loc[df_master['primary'] == primary_event].dropna().reset_index()

    arr = np.asanyarray(df1.groupby(['secondary']).nunique().sort_values(by=['index'], ascending=False).index)
    arr_tertiary = np.asanyarray(df1.groupby(['tertiary']).nunique().sort_values(by=['index'], ascending=False).index)
    
    if df1.shape[0] != 0:
        # ============= SECONDARY LABEL TRAINING ============= #
        features = df1.iloc[:, -(df1.shape[1] - 5):-1]
        label = df1['secondary']

        x_train, x_test, y_train, y_test = train_test_split(features, label, test_size=0.05, random_state=42)
        idx = x_test.index
        
        # print(y_test)
        # display(df1[df1.index.isin(idx)])
        
        with open(f'Classifiers/Transmission_PMU/Logit_Secondary_Under_{primary_event}.pickle', 'rb') as handle:
            lg = pickle.load(handle)
            
        globals()['Results_Logit_%s' % primary_event] = lg.predict_proba(x_test)
        globals()['Results_Logit_%s' % primary_event] = pd.DataFrame(globals()['Results_Logit_%s' % primary_event], columns=lg.classes_)
        globals()['Results_Logit_%s' % primary_event]['Predicted'] = np.asanyarray(globals()['Results_Logit_%s' % primary_event].idxmax(axis=1))
        globals()['Results_Logit_%s' % primary_event]['Actual'] = np.asanyarray(y_test)
        globals()['Results_Logit_%s' % primary_event]['Probability'] = np.asanyarray(globals()['Results_Logit_%s' % primary_event].max(axis=1))
        # all_pred_sec = globals()['Results_Logit_%s' % primary_event]['Predicted'].value_counts().index.to_list()
        
        acc.append(globals()['Results_Logit_%s' % primary_event][globals()['Results_Logit_%s' % primary_event]['Actual'] == globals()['Results_Logit_%s' % primary_event]['Predicted']].shape[0] / globals()['Results_Logit_%s' % primary_event].shape[0] * 100)
        
frames = [Results_Logit_Events, Results_Logit_State, Results_Logit_Equipment, 
          Results_Logit_Conditions, Results_Logit_Phase]
Results_Logit_Sec = pd.concat(frames)
Results_Logit_Sec.reset_index(inplace=True)

# Print out accuracy depending on if it is weighted after or before concatenation 
# of individual Primary label results
print(f'Accuracy of logit for Secondary level with discrimination of Primary labels: {np.mean(acc)}%')
print(f'Accuracy of logit for Secondary level w/out discrimination of Primary labels: {Results_Logit_Sec[Results_Logit_Sec.Actual == Results_Logit_Sec.Predicted].shape[0] / Results_Logit_Sec.shape[0] * 100}%')
display(Results_Logit_Sec[['Actual', 'Predicted']])

Accuracy of logit for Secondary level with discrimination of Primary labels: 85.61984830114467%
Accuracy of logit for Secondary level w/out discrimination of Primary labels: 85.98999285203716%


,Actual,Predicted
0,Vegetation,System-Event-Condition
1,System-Event-Condition,System-Event-Condition
2,External,Equipment-Failure
3,System-Event-Condition,External
4,Switching-Event,System-Event-Condition
...,...,...
1394,Single-phase-ground,Single-phase-ground
1395,Single-phase-ground,Single-phase-ground
1396,Single-phase-ground,Single-phase-ground
1397,Two-phase,Single-phase-ground


In [34]:
# Secondary evttags that only has one tertiary evttag, at least in Providers 9 and 10 
arr_unique = np.unique(one_tertiary_evttag)

In [48]:
Results_Logit_Events['Predicted'].value_counts().index.to_list()

['System-Event-Condition',
 'External',
 'Equipment-Failure',
 'Construction-&-Maintenance',
 'Switching-Event',
 'Vegetation',
 'Fault']